# 🎓 教师面板 | Teacher Dashboard

成绩查看 · 统计分析 · 可视化 · 学生管理（名册）

> 使用前：管理员需在主机端运行 `collect_grades.py` 汇总学生成绩（或已设定时任务）。本面板读取 `~/grades/all_results.json`。


In [ ]:
import sys, os, json
sys.path.insert(0, '/home/jovyan/labs')
import ipywidgets as widgets
from IPython.display import display, clear_output
from common import gradebook
from common import student_admin as sa

COLLECTED = '/home/jovyan/grades/all_results.json'
print('面板就绪。数据文件:', '存在' if os.path.exists(COLLECTED) else '未找到（先运行汇总脚本，仅可做学生管理）')


## 一、查看成绩 | View Grades

点击按钮，列出全班每生每实验得分与平均分。


In [ ]:
out_grades = widgets.Output()
btn_grades = widgets.Button(description='📋 查看成绩', button_style='primary')

def on_grades(_):
    with out_grades:
        clear_output()
        if not os.path.exists(COLLECTED):
            print('暂无汇总数据。'); return
        table = gradebook.build_grade_table(gradebook.load_collected(COLLECTED))
        if not table:
            print('暂无学生成绩。'); return
        print(f"{'学生':<16}{'完成数':>6}{'平均分':>8}")
        print('-' * 34)
        for user, row in sorted(table.items()):
            print(f"{user:<16}{row['n_done']:>6}{row['total']:>8}")
        print('\n（明细见统计部分）')
btn_grades.on_click(on_grades)
display(btn_grades, out_grades)


## 二、成绩统计与可视化 | Statistics & Charts

全班均分/中位数/及格率 + 三张图：分数分布、每实验均分、学生排名。


In [ ]:
out_stats = widgets.Output()
btn_stats = widgets.Button(description='📊 成绩统计与图表', button_style='success')

def on_stats(_):
    with out_stats:
        clear_output()
        if not os.path.exists(COLLECTED):
            print('暂无汇总数据。'); return
        table = gradebook.build_grade_table(gradebook.load_collected(COLLECTED))
        st = gradebook.statistics(table)
        print(f"提交人数: {st['n_submitted']}/{st['n_students']}  均分: {st['mean']}  "
              f"中位数: {st['median']}  标准差: {st['stdev']}  及格率: {st['pass_rate']}%")
        print('\n各实验（均分/提交数/及格率）：')
        for exp, v in sorted(st['per_experiment'].items()):
            print(f"  {exp:<28} {v['mean']:>5}  n={v['n']:<3} 及格率 {v['pass_rate']}%")
        from IPython.display import Image
        for fn in [gradebook.chart_distribution(table),
                   gradebook.chart_per_experiment(st),
                   gradebook.chart_ranking(table)]:
            if fn: display(Image(filename=fn))
btn_stats.on_click(on_stats)
display(btn_stats, out_stats)


## 三、学生管理（名册）| Student Roster

平台账号由学生用学校邮箱自助注册；此处管理**班级名册**（学号/班级/状态）。


In [ ]:
# 添加单个学生
w_email = widgets.Text(description='邮箱前缀', placeholder='如 zhangsan')
w_name = widgets.Text(description='姓名')
w_sid = widgets.Text(description='学号')
w_class = widgets.Text(description='班级')
out_add = widgets.Output()
btn_add = widgets.Button(description='➕ 添加学生', button_style='info')

def on_add(_):
    with out_add:
        clear_output()
        if not w_email.value.strip():
            print('请填邮箱前缀'); return
        s = sa.add_student(w_email.value, w_name.value, w_sid.value, w_class.value)
        print('已添加/已存在:', s['username'], s['email'])
btn_add.on_click(on_add)
display(widgets.VBox([w_email, w_name, w_sid, w_class, btn_add, out_add]))


In [ ]:
# 批量导入CSV（列：email,name,student_id,class_name，可用中文表头：邮箱,姓名,学号,班级）
w_csv = widgets.Textarea(description='CSV粘贴', layout=widgets.Layout(width='90%', height='120px'),
                         placeholder='email,name,student_id,class_name\nzhangsan@zhjpec.edu.cn,张三,20230101,汽修2301')
out_imp = widgets.Output()
btn_imp = widgets.Button(description='📥 批量导入', button_style='info')

def on_imp(_):
    with out_imp:
        clear_output()
        ok, skip, errs = sa.import_students_csv(w_csv.value)
        print(f'导入成功 {ok} 人，跳过（重复）{skip} 人')
        for e in errs: print('⚠️', e)
btn_imp.on_click(on_imp)
display(widgets.VBox([w_csv, btn_imp, out_imp]))


In [ ]:
# 名册查看与归档（已结课）
out_list = widgets.Output()
btn_list = widgets.Button(description='👥 查看名册', button_style='primary')
w_arch = widgets.Text(description='归档用户名', placeholder='输入username后点归档')
btn_arch = widgets.Button(description='🗄️ 归档（已结课）', button_style='warning')
out_arch = widgets.Output()

def on_list(_):
    with out_list:
        clear_output()
        st = sa.roster_stats()
        print(f"总人数 {st['total']}（在读 {st['active']} / 已归档 {st['archived']}）班级分布: {st['classes']}")
        for s in sa.list_students():
            print(f"  {s['username']:<16}{s.get('name',''):<8}{s.get('student_id',''):<12}{s.get('class_name',''):<12}{s['status']}")
def on_arch(_):
    with out_arch:
        clear_output()
        s = sa.archive_student(w_arch.value.strip())
        print('已归档:', s['username'] if s else '未找到该学生')
btn_list.on_click(on_list)
btn_arch.on_click(on_arch)
display(widgets.VBox([btn_list, out_list, w_arch, btn_arch, out_arch]))


---

### 说明 | Notes
- 成绩判据来自 `common/evaluator.py`（阈值=真实引擎实测值），60分及格。
- 归档仅改名册状态，保留学生信息与成绩关联，随时可恢复（`unarchive_student`）。
- 汇总脚本：主机端 `/opt/ievlab/deploy/collect_grades.py`，建议cron每小时运行。
